In [14]:
import os

from pathlib import Path
from typing import TypedDict, List, Annotated, Literal, Optional
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate

In [11]:
# Schema
class PaperSummary(BaseModel):
    title: str = Field(...,description="The official title of the paper")
    problem_domain: str = Field(..., description="The specific problem being solved (e.g., Tabular Data Classification, Time Series Forecasting).")
    authors: List[str] = Field(..., description="List of the paper's authors.")
    publication_year: Optional[int] = Field(default=None, description="The year the paper was published.")
    code_repository_url: Optional[str] = Field(
        default=None, 
        description="URL to the GitHub or code repository if provided in the text."
    )
    
    novel_architecture_proposed: Optional[str] = Field(
        default=None,
        description="Name of the new model proposed (e.g., FT-Transformer) i.e the originality of this paper. Return null if it's a survey or comparative study."
    )
    models_evaluated: List[str] = Field(..., description="List of all models tested or compared (e.g., XGBoost, ResNet, TabNet).")
    datasets_used: List[str] = Field(..., description="List of specific datasets used for benchmarking (e.g., Adult, Higgs, California Housing).")
    
    evaluation_metrics: List[str] = Field(..., description="Metrics used to measure performance (e.g., RMSE, Accuracy, F1-score).")
    
    main_conclusion: str = Field(..., description="The primary finding or takeaway of the research.")
    key_limitations: Optional[str] = Field(
        default=None, 
        description="Any stated limitations, failures, or disadvantages mentioned by the authors."
    )
    research_gap_addressed: Optional[str] = Field(
        default=None, 
        description="The specific flaw or missing piece in existing literature this paper attempts to solve (e.g., 'Previous time-series forecasting models fail to capture long-term dependencies.')."
    )
    future_research_gaps: Optional[str] = Field(
        default=None, 
        description="Unsolved problems, open questions, or future work explicitly mentioned by the authors at the end of the paper."
    )




In [20]:
llm = ChatOpenRouter(
    model="cohere/north-mini-code:free",
    temperature=0
)

In [21]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I assist you with today?', additional_kwargs={'reasoning_content': 'The user just says "Hi". We need to respond as a helpful assistant. There\'s no context. We can greet back and ask how we can help.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user just says "Hi". We need to respond as a helpful assistant. There\'s no context. We can greet back and ask how we can help.'}]}, response_metadata={'model_name': 'cohere/north-mini-code:free', 'id': 'gen-1784622593-ZtC599ivzr93biyTnV6u', 'created': 1784622593, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--019f83cb-b2f6-7141-96ad-20b5bf30406a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1, 'output_tokens': 41, 'total

In [22]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert AI researcher analyzing scientific literature. 
Your task is to extract highly accurate, factual information from the provided research paper text into the required structured format.

STRICT RULES:
1. ONLY extract information explicitly stated in the provided text.
2. DO NOT hallucinate, infer, or guess fields (especially datasets, metrics, and models).
3. If a field (like a code repository or future gap) is not mentioned in the text, leave it null/empty.
4. Keep textual summaries concise, technical, and objective."""),
    ("human", "Here is the text from the research paper:\n\n{paper_text}\n\nExtract the requested information.")
])

In [23]:
structured_llm=llm.with_structured_output(PaperSummary)

In [24]:
chain=prompt|structured_llm

In [25]:
sample_text = """
Title: Tabular Deep Learning: A Benchmark
Authors: Jane Doe, John Smith
In this paper, we address tabular data classification. We introduce a novel architecture called TabNet-V2. 
We compare it against XGBoost and Random Forest on the Adult and Higgs datasets using Accuracy and RMSE. 
While TabNet-V2 achieved higher accuracy, its training latency is a major limitation. 
Future work should focus on optimizing the attention mechanisms. Code is available at github.com/janedoe/tabnetv2.
"""

# Run the chain
result = chain.invoke({"paper_text": sample_text})

In [28]:
print(result.model_dump_json(indent=2))

{
  "title": "Tabular Deep Learning: A Benchmark",
  "problem_domain": "tabular data classification",
  "authors": [
    "Jane Doe",
    "John Smith"
  ],
  "publication_year": null,
  "code_repository_url": "github.com/janedoe/tabnetv2",
  "novel_architecture_proposed": "TabNet-V2",
  "models_evaluated": [
    "XGBoost",
    "Random Forest"
  ],
  "datasets_used": [
    "Adult",
    "Higgs"
  ],
  "evaluation_metrics": [
    "Accuracy",
    "RMSE"
  ],
  "main_conclusion": "While TabNet-V2 achieved higher accuracy, its training latency is a major limitation.",
  "key_limitations": "its training latency is a major limitation",
  "research_gap_addressed": null,
  "future_research_gaps": "Future work should focus on optimizing the attention mechanisms."
}
